# Tables review

Turns raw downloads into tidy tables, one step at a time. **This notebook so far: section 1 only** —
the raw data behind each decision that has to be made *before* `scripts/build_tables.py` is written.
Nothing here drops, cleans, or recodes anything, with two exceptions, both applied right after
loading: placeholder person IDs → real IDs from a hand table (decision 1.7, evidence §1.7), and undated
(1900-01-01) vote actions → their council meeting's date (decision 1.8, evidence §1.8).

| § | Decision | What is shown |
|---|---|---|
| 1.1 | Which source says who the alder of each ward is now | eLMS `/person`, Data Portal ward offices, DataMade memberships — per ward, side by side |
| 1.2 | Start date for "years on council" | every DataMade council membership record for each current alder |
| 1.3 | Vote values | every raw value, counts by year, every Rising Vote / Vacant / Recused row |
| 1.4 | Absence metric | attendance roll-call coverage; votes linked to attendance by meeting |
| 1.5 | Split roll call threshold | distribution of Nay counts |
| 1.6 | Which vote events are full-council votes | `actionByName` × number of voters × is the meeting a council meeting |
| 1.7 | Placeholder person IDs (13 new alders, May 2023) | every placeholder row; the name → real-ID table with its checks. **Applied at load.** |
| 1.8 | Undated (1900-01-01) actions | all 24; vote actions get the linked meeting's date. **Applied at load.** |

Sources: `data/raw/elms/` (`scripts/fetch_elms.py`), `data/raw/layers/ward_offices.geojson`
(`scripts/fetch_layers.py`), DataMade SQLite `data/raw/chicago_council.db`.
Kernel: **Chicago Council**.

In [ ]:
from pathlib import Path
import json
import sqlite3

import pandas as pd
import geopandas as gpd

# --- Settings (every choice made in this notebook) --------------------------

ELMS_DIR = Path("../data/raw/elms")
LAYERS_DIR = Path("../data/raw/layers")
DATAMADE_DB_PATH = Path("../data/raw/chicago_council.db")

# Same rule as notebooks/source_comparison.ipynb §3: eLMS dates are UTC
# timestamps, converted to a Chicago calendar date.
LOCAL_TIMEZONE = "America/Chicago"

# "Today" for deciding which DataMade memberships are current (end date on or after this).
AS_OF_DATE = "2026-10-04"

# DataMade's organization name for the full council (from its `organization` table).
DATAMADE_COUNCIL_ORGANIZATION = "Chicago City Council"

# Absence metric (decision 1.4, decided 2026-10-04): an alder is "absent from" a council meeting
# if Absent on MORE THAN this share of that meeting's roll-call items. Exactly 50% counts as present.
ABSENT_MEETING_SHARE_THRESHOLD = 0.5

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 80)

## Load raw data

eLMS item files: every action that has per-member votes becomes one row of `elms_events`, and every
member vote one row of `elms_member_votes`. All rows kept, whatever the body or date.

In [ ]:
elms_people = pd.DataFrame(json.loads((ELMS_DIR / "person_list.json").read_text())["data"])

meeting_rows, attendance_rows = [], []
for meeting_file in sorted((ELMS_DIR / "meetings").glob("*.json")):
    meeting = json.loads(meeting_file.read_text())
    roll_calls = meeting.get("attendance") or []
    meeting_rows.append({
        "meeting_id": meeting["meetingId"],
        "meeting_date_utc": meeting["date"],
        "body": meeting["body"],
        "attendance_roll_call_count": len(roll_calls),
        "attendance_names": [roll_call.get("name") for roll_call in roll_calls],
        "attendance_member_count": sum(len(roll_call.get("votes") or []) for roll_call in roll_calls),
    })
    for roll_call in roll_calls:
        for member in roll_call.get("votes") or []:
            attendance_rows.append({"meeting_id": meeting["meetingId"], "roll_call_name": roll_call.get("name"),
                                    "person_id": member["personId"], "voter_name": member["voterName"],
                                    "attendance": member["vote"]})
elms_meetings = pd.DataFrame(meeting_rows)
elms_meetings["meeting_date"] = (pd.to_datetime(elms_meetings.meeting_date_utc, utc=True)
                                 .dt.tz_convert(LOCAL_TIMEZONE).dt.strftime("%Y-%m-%d"))
elms_attendance = pd.DataFrame(attendance_rows)

event_rows, member_vote_rows = [], []
for matter_file in sorted((ELMS_DIR / "matters").glob("*.json")):
    matter = json.loads(matter_file.read_text())
    for action in matter.get("actions") or []:
        votes = action.get("votes") or []
        if not votes:
            continue
        event_id = f"{matter['matterId']}_{action['historyId']}"
        event_rows.append({
            "event_id": event_id, "matter_id": matter["matterId"], "record_number": matter["recordNumber"],
            "title": matter["title"], "action_date_utc": action["actionDate"],
            "action_by": action["actionByName"], "action_name": action["actionName"],
            "meeting_id": action["meetingId"], "voter_count": len(votes),
        })
        for member in votes:
            member_vote_rows.append({"event_id": event_id, "person_id": member["personId"],
                                     "voter_name": member["voterName"], "vote": member["vote"]})
elms_events = pd.DataFrame(event_rows)
elms_events["action_date"] = (pd.to_datetime(elms_events.action_date_utc, utc=True)
                              .dt.tz_convert(LOCAL_TIMEZONE).dt.strftime("%Y-%m-%d"))
elms_member_votes = pd.DataFrame(member_vote_rows)

ward_offices = gpd.read_file(LAYERS_DIR / "ward_offices.geojson").drop(columns="geometry")

with sqlite3.connect(DATAMADE_DB_PATH) as connection:
    datamade_council_memberships = pd.read_sql_query("""
        SELECT m.person_id AS datamade_person_id, m.person_name, post.label AS post_label, m.role,
               m.start_date, m.end_date, m.extras
        FROM membership m
        JOIN organization o ON o.id = m.organization_id
        LEFT JOIN post ON post.id = m.post_id
        WHERE o.name = ?
        ORDER BY m.person_name, m.start_date
    """, connection, params=[DATAMADE_COUNCIL_ORGANIZATION])

for name, frame in [("elms_people", elms_people), ("elms_meetings", elms_meetings),
                    ("elms_attendance", elms_attendance), ("elms_events", elms_events),
                    ("elms_member_votes", elms_member_votes), ("ward_offices", ward_offices),
                    ("datamade_council_memberships", datamade_council_memberships)]:
    print(f"{name}: {len(frame):,} rows; columns: {list(frame.columns)}")

## Placeholder person IDs → real IDs (decision 1.7, applied here so every later section uses it)

eLMS records the 13 alders first seated in May 2023 under the placeholder personId
`00000000-0000-0000-0000-000000000000` at the term's first meetings (2023-05-24, -25, -31), with
their names spelled out correctly. Decided 2026-10-04: swap in each one's real personId, matching by
name through the hand table below. The evidence for every row of the table is in **§1.7**.

- The original value stays in `person_id_raw`.
- `person_id_source`: `as recorded` / `name match` (replaced from the table) /
  `placeholder, unmatched` (placeholder, and the name isn't in the table).
- "Col?n, Rey" (4 votes, 2011, before this term) was added to the table later (2026-10-04): the person list
  has him as "Colon, Rey"; the exact-name check missed him only because of the broken "ó".

In [ ]:
PLACEHOLDER_PERSON_ID = "00000000-0000-0000-0000-000000000000"

# voterName as written on the placeholder rows -> real eLMS personId (decided 2026-10-04; evidence in §1.7).
# Every name but Fuentes and Col?n matches the person-list displayName exactly AND is used elsewhere with this ID.
PLACEHOLDER_PERSON_ID_BY_NAME = {
    "Chico, Peter":            "4C168A10-6BF3-ED11-A7C6-001DD804FC2B",  # ward 10
    "Clay, Angela":            "5F4C907D-6CF3-ED11-A7C6-001DD804FC2B",  # ward 46
    "Conway, William":         "4C380420-6CF3-ED11-A7C6-001DD804FC2B",  # ward 34
    "Cruz, Ruth":              "C7422EFD-6BF3-ED11-A7C6-001DD804FC2B",  # ward 30
    "Fuentes, Jessica":        "85DA45DD-6BF3-ED11-A7C6-001DD804FC2B",  # ward 26; listed as "Fuentes, Jessica L."
    "Gutierrez, Jeylu B.":     "2E2DFB80-6BF3-ED11-A7C6-001DD804FC2B",  # ward 14
    "Hall, William E.":        "7D95D6BB-6AF3-ED11-A7C6-001DD804FC2B",  # ward 06
    "Lawson, Bennett R.":      "1628D550-6CF3-ED11-A7C6-001DD804FC2B",  # ward 44
    "Manaa-Hoppenworth, Leni": "561B99CA-6CF3-ED11-A7C6-001DD804FC2B",  # ward 48
    "Mosley, Ronnie L.":       "8B0682AE-6BF3-ED11-A7C6-001DD804FC2B",  # ward 21
    "Ramirez, Julia M.":       "9E248041-6BF3-ED11-A7C6-001DD804FC2B",  # ward 12
    "Robinson, Lamont J.":     "80D52B16-6AF3-ED11-A7C6-001DD804FC2B",  # ward 04
    "Yancy, Desmon C.":        "E0AC4B85-6AF3-ED11-A7C6-001DD804FC2B",  # ward 05
    # Added 2026-10-04 (user): 4 votes in 2011, before this term. "?" is a broken "ó"; the
    # person list has him as "Colon, Rey" (ward 35, inactive) and no other Colon/Colón.
    "Col?n, Rey":              "184EE8BE-ABFD-ED11-8847-001DD8068005",  # ward 35
}

def replace_placeholder_ids(frame):
    """Keep the original ID in person_id_raw, and replace placeholder IDs whose voter_name is in the table."""
    frame = frame.copy()
    frame["person_id_raw"] = frame.person_id
    is_placeholder = frame.person_id == PLACEHOLDER_PERSON_ID
    replacement_id = frame.voter_name.map(PLACEHOLDER_PERSON_ID_BY_NAME)
    is_replaced = is_placeholder & replacement_id.notna()
    frame.loc[is_replaced, "person_id"] = replacement_id[is_replaced]
    frame["person_id_source"] = "as recorded"
    frame.loc[is_replaced, "person_id_source"] = "name match"
    frame.loc[is_placeholder & ~is_replaced, "person_id_source"] = "placeholder, unmatched"
    return frame

elms_member_votes = replace_placeholder_ids(elms_member_votes)
elms_attendance = replace_placeholder_ids(elms_attendance)
for name, frame in [("elms_member_votes", elms_member_votes), ("elms_attendance", elms_attendance)]:
    print(f"{name} person_id_source:", frame.person_id_source.value_counts().to_dict())

## Undated actions (1900-01-01) → meeting date (decision 1.8, applied here; evidence in §1.8)

eLMS writes `actionDate = 1900-01-01` when no date was entered. Decided 2026-10-04:

- **Vote actions** (the only actions in `elms_events`): if the action's `meetingId` is a City Council
  meeting file, use that meeting's Chicago date. `action_date_source` = `meeting date`.
- If there's no such meeting, the date is left missing, `action_date_source` = `missing (1900 placeholder)`.
- Every other action keeps its date, `action_date_source` = `eLMS`.
- Non-vote actions with this date (14) aren't loaded into any table yet. §1.8 lists them, and the same
  rule (leave missing, flag) carries into `build_tables.py`.

In [ ]:
# Raw eLMS timestamps that mean "no date entered". Same constant and rule as
# notebooks/source_comparison.ipynb settings (identified by the user 2026-09-30 as entry errors).
ELMS_PLACEHOLDER_TIMESTAMPS = ["1900-01-01T00:00:00+00:00"]

elms_events["action_date_raw"] = elms_events.action_date_utc
elms_events["action_date_source"] = "eLMS"
is_undated = elms_events.action_date_utc.isin(ELMS_PLACEHOLDER_TIMESTAMPS)
linked_meeting_date = elms_events.meeting_id.map(elms_meetings.set_index("meeting_id").meeting_date)
has_meeting_date = is_undated & linked_meeting_date.notna()
elms_events.loc[has_meeting_date, "action_date"] = linked_meeting_date[has_meeting_date]
elms_events.loc[has_meeting_date, "action_date_source"] = "meeting date"
elms_events.loc[is_undated & ~has_meeting_date, "action_date"] = None
elms_events.loc[is_undated & ~has_meeting_date, "action_date_source"] = "missing (1900 placeholder)"
print("elms_events action_date_source:", elms_events.action_date_source.value_counts().to_dict())

## 1.1 Who is the alder of each ward now?

Three sources, one row per ward, all shown as recorded (names not cleaned):
- **eLMS `/person`**: one record per person with a single `ward` value and `isActive`. No dates.
- **Data Portal ward offices**: `alderman` name per ward; `photo_link` file name is an eLMS personId.
- **DataMade memberships** whose end date is on or after `AS_OF_DATE`.

First every eLMS person record (123), so the odd ones are visible: inactive people are kept, plus
Clerk / Mayor rows and a "ward 68".

In [ ]:
elms_people_view = elms_people[["ward", "displayName", "isActive", "personId", "lastPublicationDate"]].copy()
elms_people_view["name_has_trailing_space"] = elms_people_view.displayName.str.endswith(" ")
display(elms_people_view.sort_values(["ward", "isActive", "displayName"]).reset_index(drop=True))
print("ward values:", sorted(elms_people.ward.unique(), key=str))
print("isActive counts:", elms_people.isActive.value_counts().to_dict())

In [ ]:
active_by_ward = (elms_people[elms_people.isActive]
                  .groupby("ward").agg(elms_active_names=("displayName", list),
                                       elms_active_person_ids=("personId", list)))

ward_offices_view = ward_offices[["ward", "alderman", "photo_link"]].copy()
ward_offices_view["ward"] = ward_offices_view.ward.astype(int).map("{:02d}".format)
# photo_link is stored as {"url": "..."} (a dict, or its JSON text depending on the reader).
ward_offices_view["photo_url"] = [link["url"] if isinstance(link, dict) else json.loads(link)["url"]
                                  for link in ward_offices_view.photo_link]
ward_offices_view["photo_file_name"] = ward_offices_view.photo_url.str.split("/").str[-1]
ward_offices_view = ward_offices_view.set_index("ward")

current_datamade = datamade_council_memberships[datamade_council_memberships.end_date >= AS_OF_DATE].copy()
current_datamade["ward"] = current_datamade.post_label.str.replace("Ward ", "").astype(int).map("{:02d}".format)
datamade_by_ward = current_datamade.groupby("ward").agg(datamade_current_names=("person_name", list),
                                                        datamade_start_dates=("start_date", list))

ward_index = pd.Index([f"{ward:02d}" for ward in range(1, 51)], name="ward")
current_alder_sources = (pd.DataFrame(index=ward_index)
                         .join(active_by_ward).join(ward_offices_view[["alderman", "photo_file_name"]])
                         .join(datamade_by_ward))
current_alder_sources["elms_active_count"] = current_alder_sources.elms_active_names.str.len()
current_alder_sources["datamade_current_count"] = current_alder_sources.datamade_current_names.str.len()
# Does the ward-office photo file name contain the active eLMS personId? (photo_link naming, 2026-09-30 notes)
current_alder_sources["photo_matches_elms_active_id"] = [
    isinstance(ids, list) and isinstance(photo, str) and any(person_id in photo for person_id in ids)
    for ids, photo in zip(current_alder_sources.elms_active_person_ids, current_alder_sources.photo_file_name)
]
display(current_alder_sources.drop(columns=["elms_active_person_ids"]))

In [ ]:
# Wards where the sources do not give exactly one name each, or the photo id does not match.
ward_disagreements = current_alder_sources[
    (current_alder_sources.elms_active_count != 1)
    | (current_alder_sources.datamade_current_count != 1)
    | ~current_alder_sources.photo_matches_elms_active_id
]
print(f"wards needing a look: {len(ward_disagreements)}")
display(ward_disagreements.drop(columns=["elms_active_person_ids"]))

**Decision 1.1** — which source defines the current alder per ward (`CURRENT_ALDER_SOURCE`), and how
to handle any ward listed above (`ALDER_OVERRIDES`, a hand table with a note per row).
Name matching between eLMS and DataMade (needed for 1.2) is by name text — see the next cell.

## 1.2 Start date for "years on council"

Agreed definition: **total years on the council**, across every ward the person has represented.
eLMS has no dates, so the candidate source is DataMade's council membership records. Below: every
DataMade council membership record (all years, all people) for the names of currently active eLMS
alders.

**How names are matched here:** eLMS `displayName` with trailing spaces stripped, compared to DataMade
`person_name` exactly. This is only for display; anyone not matched is listed in the next cell.

Things to check by eye: gaps between records for the same person (e.g. Reilly, Sposato); whether
`post_label` is the ward held *at that time* or a later ward number; duplicate people
(e.g. "Burnett, Jr., Walter" and "Burnett, Walter R." — one person or two?); spelling variants
("Colon" / "Colón").

In [ ]:
active_elms_names = elms_people[elms_people.isActive & elms_people.ward.str.isdigit()].copy()
active_elms_names["name_stripped"] = active_elms_names.displayName.str.strip()

current_alder_memberships = (datamade_council_memberships
                             .merge(active_elms_names[["name_stripped", "ward"]],
                                    left_on="person_name", right_on="name_stripped", how="inner")
                             .rename(columns={"ward": "elms_current_ward"})
                             .drop(columns=["name_stripped"]))
current_alder_memberships["records_for_person"] = current_alder_memberships.groupby("person_name").person_name.transform("size")
display(current_alder_memberships.sort_values(["elms_current_ward", "start_date"]).reset_index(drop=True))

unmatched = active_elms_names[~active_elms_names.name_stripped.isin(datamade_council_memberships.person_name)]
print(f"active eLMS alders with no exact-name DataMade council record: {len(unmatched)}")
display(unmatched[["ward", "displayName", "personId"]])

In [ ]:
# People with more than one DataMade council record (possible gaps, re-appointments, ward changes).
multi_record_people = datamade_council_memberships[
    datamade_council_memberships.groupby("person_name").person_name.transform("size") > 1]
display(multi_record_people)
print("distinct values of `role`:", datamade_council_memberships.role.unique().tolist())
print("distinct values of `extras`:", datamade_council_memberships.extras.unique().tolist())

**Decision 1.2** — use DataMade start dates as-is, or with a hand table (`TENURE_START_OVERRIDES`, one row
per correction with a citation). OPEN QUESTION for each multi-record person: was there a real break in
service, or is the gap a data artifact? (DataMade's `merge_memberships.sql` merges back-to-back records —
2026-09-30 notes.)

## 1.3 Vote values — kept as recorded

Every raw value, by year of the action date. Then every row for the rare values.

In [ ]:
member_votes_with_dates = elms_member_votes.merge(elms_events[["event_id", "action_date", "record_number", "title",
                                                               "action_by", "action_name"]], on="event_id")
member_votes_with_dates["year"] = member_votes_with_dates.action_date.str[:4]
display(pd.crosstab(member_votes_with_dates.vote, member_votes_with_dates.year, margins=True))

In [ ]:
RARE_VOTE_VALUES = ["Rising Vote", "Vacant", "Recused"]   # listed in full; display only
for vote_value in RARE_VOTE_VALUES:
    rows = member_votes_with_dates[member_votes_with_dates.vote == vote_value]
    summary = (rows.groupby(["action_date", "record_number", "title", "action_by", "action_name"])
               .agg(member_count=("person_id", "size"), voters=("voter_name", lambda names: "; ".join(names)))
               .reset_index())
    print(f"=== {vote_value}: {len(rows)} member votes on {len(summary)} events")
    display(summary)

Note: all Rising Vote records are on one event (shown above) — check its date against the term.

## 1.4 Absence — what the data can show

**(a) Attendance roll calls.** Each council meeting detail file has an `attendance` list. Below: every
council meeting, with how many roll calls and members it has.

In [ ]:
council_meetings = elms_meetings[elms_meetings.body == "City Council"].sort_values("meeting_date")
display(council_meetings[["meeting_date", "meeting_id", "attendance_roll_call_count", "attendance_names",
                          "attendance_member_count"]].reset_index(drop=True))
print("meetings by attendance member count:", council_meetings.attendance_member_count.value_counts().to_dict())
print("attendance values:", elms_attendance.attendance.value_counts().to_dict())

**(b) Votes linked to attendance.** Every vote action has a `meetingId`, so each member's votes at a
meeting can be set next to that member's attendance status. One row per person × meeting:
attendance status, and how many vote events they had of each raw vote value. Shown: the counts of
each (attendance status, vote-value pattern) combination, then example rows of the mixed ones.

In [ ]:
votes_per_person_meeting = (member_votes_with_dates
                            .merge(elms_events[["event_id", "meeting_id"]], on="event_id")
                            .pivot_table(index=["meeting_id", "person_id"], columns="vote",
                                         values="event_id", aggfunc="count", fill_value=0)
                            .reset_index())
vote_value_columns = [column for column in votes_per_person_meeting.columns if column not in ("meeting_id", "person_id")]
person_meeting = votes_per_person_meeting.merge(
    elms_attendance[["meeting_id", "person_id", "attendance"]], on=["meeting_id", "person_id"], how="left")
person_meeting["attendance"] = person_meeting.attendance.fillna("(no attendance record)")
person_meeting["total_events"] = person_meeting[vote_value_columns].sum(axis=1)
person_meeting["pattern"] = [
    "all Absent" if row.Absent == row.total_events else ("no Absent" if row.Absent == 0 else "some Absent")
    for row in person_meeting.itertuples()]
display(pd.crosstab(person_meeting.attendance, person_meeting.pattern, margins=True))

In [ ]:
person_names = elms_people.set_index("personId").displayName
person_meeting["name"] = person_meeting.person_id.map(person_names)
person_meeting = person_meeting.merge(elms_meetings[["meeting_id", "meeting_date"]], on="meeting_id", how="left")
show_columns = ["meeting_date", "name", "attendance", "total_events"] + vote_value_columns
print("Marked Present at roll call, but Absent on some or all vote events:")
display(person_meeting[(person_meeting.attendance == "Present") & (person_meeting.pattern != "no Absent")]
        .sort_values(["meeting_date", "name"])[show_columns].reset_index(drop=True))
print("Marked Absent at roll call, but recorded voting on some events:")
display(person_meeting[(person_meeting.attendance == "Absent") & (person_meeting.pattern != "all Absent")]
        .sort_values(["meeting_date", "name"])[show_columns].reset_index(drop=True))

In [ ]:
# Check of decision 1.7 (placeholder IDs): rows at the two affected council meetings.
# Expect 50 alders per meeting, no placeholder person_id, and a name for every row.
PLACEHOLDER_CHECK_DATES = ["2023-05-24", "2023-05-31"]
def placeholder_check(frame):
    on_dates = frame[frame.meeting_date.isin(PLACEHOLDER_CHECK_DATES)]
    return on_dates.groupby("meeting_date").agg(
        alder_rows=("person_id", "size"), distinct_people=("person_id", "nunique"),
        placeholder_rows=("person_id", lambda ids: (ids == PLACEHOLDER_PERSON_ID).sum()),
        rows_without_name=("person_id", lambda ids: ids.map(person_names).isna().sum()))
display(placeholder_check(person_meeting))

**Decision 1.4** — which absence metric(s) to build: (a) % of council meetings marked Absent at roll call;
(b) % of vote events with vote = Absent; (later) (b) restricted to genuine roll calls. "Not Voting" is
not counted as absent (user, 2026-10-04). OPEN QUESTION: when in the meeting the attendance roll call
is taken — the tables above show how often roll call and per-item votes disagree.

## 1.4b Partial absence and lateness (display only)

Decided 2026-10-04: the main absence metric is **(e) % of meetings where the alder is Absent on every
item**. Roll call is computed but not used on the dashboard for now. Two side questions:

1. **Partial absence** — when an alder is Absent on *some* items of a meeting, was it one item or most
   of the meeting? Was there a walkout (many alders Absent on the same items)?
2. **Lateness** — roll call says Absent, but the alder voted on at least one item ("arrived late?").

Notes: only vote events with a `meetingId` that matches a meeting file are counted here (the 36
blank-meetingId events that could be matched by date are not yet matched — pending decision).

In [ ]:
partial = person_meeting[person_meeting.pattern == "some Absent"].copy()
partial["share_of_items_absent"] = (partial.Absent / partial.total_events).round(3)
print(f"alder-meetings with some (not all) items Absent: {len(partial)}")
print("Share of that meeting's items missed, in bins:")
SHARE_BINS = [0, 0.1, 0.25, 0.5, 0.75, 0.9999, 1]   # display bins only
display(pd.cut(partial.share_of_items_absent, SHARE_BINS).value_counts().sort_index().rename("alder_meetings"))
display(partial.sort_values(["meeting_date", "share_of_items_absent"], ascending=[True, False])
        [["meeting_date", "name", "attendance", "total_events", "Absent", "share_of_items_absent"]].reset_index(drop=True))

**Distribution of absent share, all alder-meetings** — for every alder at every meeting: the share of
that meeting's items on which they were Absent. Most are 0 (never Absent) or 1 (Absent on all), so the
counts are listed exactly, then the in-between values are plotted on their own.

Decided 2026-10-04: an alder counts as absent from a meeting if Absent on **more than**
`ABSENT_MEETING_SHARE_THRESHOLD` = 50% of its items; exactly 50% counts as present. (Here, before the
§1.4d rules, 64 alder-meetings sit at exactly 0.5 — two 2-item meetings; §1.4d shows the count after.)

In [ ]:
import matplotlib.pyplot as plt
person_meeting["share_of_items_absent"] = person_meeting.Absent / person_meeting.total_events
print("alder-meetings by exact share (rounded to 3 decimals):")
display(person_meeting.share_of_items_absent.round(3).value_counts().sort_index().rename("alder_meetings").to_frame())

in_between = person_meeting[(person_meeting.share_of_items_absent > 0) & (person_meeting.share_of_items_absent < 1)]
figure, axis = plt.subplots(figsize=(8, 3))
axis.hist(in_between.share_of_items_absent, bins=50, range=(0, 1), color="tab:blue")
axis.axvline(ABSENT_MEETING_SHARE_THRESHOLD, color="black", linestyle="--", linewidth=1,
             label="threshold (absent if above)")
axis.set_xlabel("share of the meeting's items marked Absent")
axis.set_ylabel("alder-meetings")
axis.set_title(f"Alder-meetings with share strictly between 0 and 1 (n={len(in_between)}); "
               f"not shown: share 0 (n={(person_meeting.share_of_items_absent == 0).sum()}), "
               f"share 1 (n={(person_meeting.share_of_items_absent == 1).sum()})", fontsize=9)
axis.legend()
plt.show()

**Partial absence by meeting** — how many alders were partially absent at each meeting. Many on one
date could mean a walkout or a long meeting people left early.

In [ ]:
display(partial.groupby("meeting_date")
        .agg(alders_partially_absent=("name", "size"),
             median_share_of_items_missed=("share_of_items_absent", "median"),
             items_at_meeting=("total_events", "max"))
        .reset_index())

**Walkout check, per item** — for each vote event: how many alders were Absent on it, compared with
how many were Absent at that meeting's roll call. Listed: events where at least
`WALKOUT_EXTRA_ABSENT` more alders are Absent on the item than at roll call (display threshold only).

In [ ]:
WALKOUT_EXTRA_ABSENT = 5   # display threshold only — not used to drop or label anything
absent_per_event = (elms_member_votes.assign(is_absent=elms_member_votes.vote == "Absent")
                    .groupby("event_id").is_absent.sum().rename("absent_on_item"))
roll_call_absent = (elms_attendance[elms_attendance.attendance == "Absent"]
                    .groupby("meeting_id").size().rename("absent_at_roll_call"))
event_absence = (elms_events[["event_id", "meeting_id", "action_date", "record_number", "title", "action_name"]]
                 .join(absent_per_event, on="event_id").join(roll_call_absent, on="meeting_id"))
event_absence["absent_at_roll_call"] = event_absence.absent_at_roll_call.fillna(0).astype(int)
event_absence["extra_absent"] = event_absence.absent_on_item - event_absence.absent_at_roll_call
walkout_candidates = event_absence[event_absence.extra_absent >= WALKOUT_EXTRA_ABSENT]
print(f"events with >= {WALKOUT_EXTRA_ABSENT} more Absent than roll call: {len(walkout_candidates)}")
display(walkout_candidates.sort_values(["action_date", "extra_absent"], ascending=[True, False])
        .drop(columns=["event_id", "meeting_id"]).reset_index(drop=True))

**Lateness** — per alder: meetings where roll call says Absent but they voted (non-Absent) on at least
one item. Shown with and without the 2023-09-13 meeting, where ~40 alders fit this pattern at once.

In [ ]:
late = person_meeting[(person_meeting.attendance == "Absent") & (person_meeting.pattern != "all Absent")]
late_per_alder = (late.groupby("name").agg(late_meetings=("meeting_date", "size"),
                                           dates=("meeting_date", lambda dates: ", ".join(sorted(dates))))
                  .sort_values("late_meetings", ascending=False))
late_per_alder["late_meetings_excluding_2023_09_13"] = (
    late[late.meeting_date != "2023-09-13"].groupby("name").size().reindex(late_per_alder.index).fillna(0).astype(int))
print(f"late alder-meetings: {len(late)}; by meeting:")
display(late.meeting_date.value_counts().sort_index().rename("late_alders"))
display(late_per_alder.reset_index())

## 1.4c The two "exactly 50%" meetings — raw records

The user asked (2026-10-04) whether the 2024-01-31 meeting is parsed correctly: press reports say
Dowell, Coleman and Burnett Jr. left before the Gaza ceasefire vote and Mitts was absent, yet the
table above shows all 50 alders "Absent" on half that meeting's items. Below: every vote action
in the raw files for the two items behind the 50% rows, with action text and vote counts.

In [ ]:
def show_raw_actions(record_number):
    """Every action (with or without votes) on one item, straight from its raw eLMS file."""
    for matter_file in (ELMS_DIR / "matters").glob("*.json"):
        matter = json.loads(matter_file.read_text())
        if matter["recordNumber"] != record_number:
            continue
        print(f"=== {record_number}: {matter['title']}")
        for action in matter["actions"]:
            votes = action.get("votes") or []
            print(f"  {action['actionDate']} | {action['actionByName']} | {action['actionName']} | meetingId={action['meetingId']}")
            print(f"    vote counts: {pd.Series([v['vote'] for v in votes]).value_counts().to_dict() if votes else '(no votes)'}")
            print(f"    actionText: {action['actionText']}")
            not_yea = [(v["voterName"], v["vote"]) for v in votes if v["vote"] != "Yea"]
            if not_yea and len(not_yea) < 50:
                print(f"    not Yea: {not_yea}")

show_raw_actions("SR2023-0006422")   # Gaza ceasefire resolution, 2024-01-31 (for comparison with press reports)
show_raw_actions("O2024-0007513")    # "Time fixed for next meeting", 2024-01-31 — the all-Absent item
show_raw_actions("O2025-0020484")    # 2025-12-18 meeting — the 14-Absent item

**Voice votes.** The all-Absent item's action text says "Passed by a voice vote", yet a 50-member
roster is attached. Every vote action whose `actionText` contains "voice vote":

In [ ]:
VOICE_VOTE_TEXT = "voice vote"   # display only — searched for in actionText (case-insensitive)
action_texts = []
for matter_file in (ELMS_DIR / "matters").glob("*.json"):
    matter = json.loads(matter_file.read_text())
    for action in matter.get("actions") or []:
        if action.get("votes"):
            action_texts.append({"event_id": f"{matter['matterId']}_{action['historyId']}", "action_text": action["actionText"]})
action_texts = pd.DataFrame(action_texts)
voice_vote_events = (elms_events.merge(action_texts, on="event_id")
                     .loc[lambda frame: frame.action_text.str.contains(VOICE_VOTE_TEXT, case=False, na=False)])
voice_vote_counts = (elms_member_votes[elms_member_votes.event_id.isin(voice_vote_events.event_id)]
                     .pivot_table(index="event_id", columns="vote", values="person_id", aggfunc="count", fill_value=0))
display(voice_vote_events[["event_id", "action_date", "record_number", "title", "action_by", "action_name", "action_text"]]
        .join(voice_vote_counts, on="event_id").drop(columns="event_id").reset_index(drop=True))

**Committee-labeled actions with 50 voters** — the vote values they carry (the 2025-12-18 item is a
committee action recorded as Present/Absent, attached to the council meeting's `meetingId`).

In [ ]:
committee_50 = elms_events[elms_events.action_by.str.startswith("Committee") & (elms_events.voter_count == 50)]
committee_50_counts = (elms_member_votes[elms_member_votes.event_id.isin(committee_50.event_id)]
                       .pivot_table(index="event_id", columns="vote", values="person_id", aggfunc="count", fill_value=0))
display(committee_50[["event_id", "action_date", "record_number", "title", "action_by", "action_name", "meeting_id"]]
        .join(committee_50_counts, on="event_id").drop(columns="event_id")
        .sort_values("action_date").reset_index(drop=True))

**Rosters that look like attendance** (user asked to list all before any rule, 2026-10-04).
First: every vote event whose roster has **no Yea and no Nay**. Then: every event with at least one
**Present** value (247 member votes in total), with its full vote counts.

In [ ]:
counts_per_event = elms_member_votes.pivot_table(index="event_id", columns="vote", values="person_id",
                                                 aggfunc="count", fill_value=0)
event_listing = (elms_events[["event_id", "action_date", "record_number", "title", "action_by", "action_name"]]
                 .merge(action_texts, on="event_id").join(counts_per_event, on="event_id"))
no_yea_or_nay = event_listing[(event_listing.Yea == 0) & (event_listing.Nay == 0)]
print(f"events with no Yea and no Nay: {len(no_yea_or_nay)}")
display(no_yea_or_nay.drop(columns="event_id").sort_values("action_date").reset_index(drop=True))

In [ ]:
with_present = event_listing[event_listing.Present > 0]
print(f"events with at least one Present: {len(with_present)}; Present member votes: {with_present.Present.sum()}")
display(with_present.drop(columns="event_id").sort_values("action_date").reset_index(drop=True))

## 1.4d Absent-share distribution, re-run with the agreed rules

Two rules agreed 2026-10-04, applied here provisionally (they move into `scripts/build_tables.py` next):

1. **`roster_kind`** — one label per vote event. Rules are checked in the order listed; the first match
   wins. Only `roll call` events count toward absence. Nothing is dropped: every non-`roll call`
   event is listed below.
2. **Match blank meetingIds by date** — only when exactly one City Council meeting falls on the
   event's Chicago date. Flagged `meeting_id_source = "date match"`.

In [ ]:
# --- roster_kind rules (first match wins; checked top to bottom) ------------
# Each rule: (label, description). The test for each is written out in assign_roster_kind() below.
ROSTER_KIND_RULES = [
    ("voice vote",           "actionText contains 'voice vote' (case-insensitive)"),
    ("vote not taken",       "actionText contains 'vote not taken' (case-insensitive)"),
    ("rising vote",          "roster contains 'Rising Vote' and no Yea/Nay"),
    ("committee attendance", "action_by starts with 'Committee' and roster has no Yea/Nay"),
    ("present roster",       "action_by is 'City Council' (or blank) and roster has no Yea/Nay"),
    ("roll call",            "everything else"),
]

def assign_roster_kind(row):
    text = (row.action_text or "").lower()
    has_yea_or_nay = (row.Yea + row.Nay) > 0
    if "voice vote" in text:
        return "voice vote"
    if "vote not taken" in text:
        return "vote not taken"
    if not has_yea_or_nay and row["Rising Vote"] > 0:
        return "rising vote"
    if not has_yea_or_nay and row.action_by.startswith("Committee"):
        return "committee attendance"
    if not has_yea_or_nay:
        return "present roster"
    return "roll call"

event_listing["roster_kind"] = event_listing.apply(assign_roster_kind, axis=1)
display(event_listing.roster_kind.value_counts().rename("events").to_frame())
display(event_listing[event_listing.roster_kind != "roll call"]
        .drop(columns=["event_id"]).sort_values(["roster_kind", "action_date"]).reset_index(drop=True))

In [ ]:
# --- Match blank meetingIds by date ------------------------------------------
MATCH_MEETING_BY_DATE = True   # decided 2026-10-04; only when exactly one council meeting that date
council_meetings_per_date = council_meetings.groupby("meeting_date").meeting_id.agg(list)
events_matched = elms_events[["event_id", "meeting_id", "action_date"]].copy()
events_matched["meeting_id_source"] = "eLMS"
blank_meeting = events_matched.meeting_id.isna() | (events_matched.meeting_id == "")
candidates = events_matched.loc[blank_meeting, "action_date"].map(council_meetings_per_date)
single_match = candidates.apply(lambda ids: isinstance(ids, list) and len(ids) == 1)
if MATCH_MEETING_BY_DATE:
    matched_index = single_match[single_match].index
    events_matched.loc[matched_index, "meeting_id"] = candidates[matched_index].str[0]
    events_matched.loc[matched_index, "meeting_id_source"] = "date match"
print("meeting_id_source counts:", events_matched.meeting_id_source.value_counts().to_dict())
print(f"blank meetingIds left unmatched: {(blank_meeting & (events_matched.meeting_id_source == 'eLMS')).sum()}")

In [ ]:
# --- Re-run: share of each meeting's roll-call items an alder was Absent on ---
roll_call_event_ids = set(event_listing.loc[event_listing.roster_kind == "roll call", "event_id"])
rerun = (elms_member_votes[elms_member_votes.event_id.isin(roll_call_event_ids)]
         .merge(events_matched[["event_id", "meeting_id"]], on="event_id")
         .loc[lambda frame: frame.meeting_id.isin(set(council_meetings.meeting_id))])
rerun_person_meeting = (rerun.assign(is_absent=rerun.vote == "Absent")
                        .groupby(["meeting_id", "person_id"])
                        .agg(item_count=("event_id", "size"), absent_items=("is_absent", "sum"))
                        .reset_index())
rerun_person_meeting["share_of_items_absent"] = rerun_person_meeting.absent_items / rerun_person_meeting.item_count
print(f"alder-meetings: {len(rerun_person_meeting)}")
print("alder-meetings by exact share (rounded to 3 decimals):")
display(rerun_person_meeting.share_of_items_absent.round(3).value_counts().sort_index().rename("alder_meetings").to_frame())
in_between_rerun = rerun_person_meeting[(rerun_person_meeting.share_of_items_absent > 0)
                                        & (rerun_person_meeting.share_of_items_absent < 1)]
display(in_between_rerun.assign(name=in_between_rerun.person_id.map(person_names))
        .merge(elms_meetings[["meeting_id", "meeting_date"]], on="meeting_id")
        .sort_values("share_of_items_absent")[["meeting_date", "name", "item_count", "absent_items", "share_of_items_absent"]]
        .reset_index(drop=True))

In [ ]:
# Check of decision 1.7 on the re-run (same check as end of §1.4).
rerun_person_meeting = rerun_person_meeting.merge(elms_meetings[["meeting_id", "meeting_date"]], on="meeting_id", how="left")
display(placeholder_check(rerun_person_meeting))

In [ ]:
# Apply the absence rule: absent from meeting if share > ABSENT_MEETING_SHARE_THRESHOLD.
rerun_person_meeting["absent_from_meeting"] = (rerun_person_meeting.share_of_items_absent
                                               > ABSENT_MEETING_SHARE_THRESHOLD)
print("absent_from_meeting counts:", rerun_person_meeting.absent_from_meeting.value_counts().to_dict())
# For reference only: alder-meetings sitting exactly at the threshold (these count as present).
at_threshold = rerun_person_meeting[rerun_person_meeting.share_of_items_absent == ABSENT_MEETING_SHARE_THRESHOLD]
print(f"alder-meetings exactly at the threshold (counted present): {len(at_threshold)}")
display(at_threshold.assign(name=at_threshold.person_id.map(person_names))
        [["meeting_date", "name", "item_count", "absent_items"]].reset_index(drop=True))

## 1.5 Split roll calls — how many Nays?

Number of vote events by number of Nay votes (all events, all bodies).

In [ ]:
nay_counts = (elms_member_votes.assign(is_nay=elms_member_votes.vote == "Nay")
              .groupby("event_id").is_nay.sum().rename("nay_count"))
nay_distribution = nay_counts.value_counts().sort_index().rename("event_count").to_frame()
nay_distribution["events_with_at_least_this_many"] = nay_distribution.event_count[::-1].cumsum()[::-1]
display(nay_distribution)

**Decision 1.5** — `SPLIT_MIN_NAY` (events with at least this many Nays count as split).

## 1.6 Which vote events are full-council votes?

`actionByName` on each vote action, crossed with whether the event has 50 voters and whether its
`meetingId` is a City Council meeting we have a file for.

In [ ]:
council_meeting_ids = set(council_meetings.meeting_id)
elms_events["has_50_voters"] = elms_events.voter_count == 50
elms_events["meeting_kind"] = [
    "City Council meeting file" if meeting_id in council_meeting_ids
    else ("blank meetingId" if not meeting_id else "meetingId with no meeting file")
    for meeting_id in elms_events.meeting_id]
display(elms_events.groupby(["action_by", "has_50_voters", "meeting_kind"]).size().rename("event_count").reset_index())

In [ ]:
print("Every event that is not (City Council, 50 voters, council meeting file):")
unusual_events = elms_events[~((elms_events.action_by == "City Council") & elms_events.has_50_voters
                               & (elms_events.meeting_kind == "City Council meeting file"))]
display(unusual_events[["action_date", "record_number", "title", "action_by", "action_name",
                        "voter_count", "meeting_kind", "meeting_id"]]
        .sort_values(["action_date", "record_number"]).reset_index(drop=True))

**Decision 1.6** — which of these count as council votes (`INCLUDE_ACTION_BY`, and what to do with a
blank `actionByName` or a missing meeting file). Nothing is excluded until this is decided.

## 1.7 Placeholder person IDs: evidence for the name table

The replacement itself happens right after loading (see "Placeholder person IDs → real IDs"). This
section shows why each row of `PLACEHOLDER_PERSON_ID_BY_NAME` is right, using `person_id_raw` (the ID as
eLMS recorded it).

In [ ]:
# Every placeholder row, by source and Chicago date.
placeholder_votes = (elms_member_votes[elms_member_votes.person_id_raw == PLACEHOLDER_PERSON_ID]
                     .merge(elms_events[["event_id", "action_date"]], on="event_id")
                     .rename(columns={"action_date": "date"}).assign(source="vote"))
placeholder_attendance = (elms_attendance[elms_attendance.person_id_raw == PLACEHOLDER_PERSON_ID]
                          .merge(elms_meetings[["meeting_id", "meeting_date"]], on="meeting_id")
                          .rename(columns={"meeting_date": "date"}).assign(source="attendance"))
placeholder_rows = pd.concat([placeholder_votes, placeholder_attendance], ignore_index=True)
display(placeholder_rows.groupby(["source", "date"]).size().rename("rows").to_frame())

In [ ]:
# One row per placeholder name: what the table maps it to, and the two independent checks.
#   rows_same_name_real_id: rows elsewhere where eLMS writes this exact voterName with that real ID
#   person_list_display_name: the eLMS person-list name for that real ID
real_id_rows = pd.concat([elms_member_votes, elms_attendance])
real_id_rows = real_id_rows[real_id_rows.person_id_raw != PLACEHOLDER_PERSON_ID]
placeholder_name_table = (placeholder_rows
                          .groupby("voter_name")
                          .agg(placeholder_rows=("source", "size"), first_date=("date", "min"),
                               last_date=("date", "max"), person_id_source=("person_id_source", "first"))
                          .reset_index())
placeholder_name_table["mapped_person_id"] = placeholder_name_table.voter_name.map(PLACEHOLDER_PERSON_ID_BY_NAME)
placeholder_name_table["rows_same_name_real_id"] = [
    ((real_id_rows.voter_name == name) & (real_id_rows.person_id == person_id)).sum()
    for name, person_id in zip(placeholder_name_table.voter_name, placeholder_name_table.mapped_person_id)]
person_list = elms_people.set_index("personId")
placeholder_name_table["person_list_display_name"] = placeholder_name_table.mapped_person_id.map(person_list.displayName)
placeholder_name_table["person_list_ward"] = placeholder_name_table.mapped_person_id.map(person_list.ward)
display(placeholder_name_table)

In [ ]:
# Fuentes: the one name without an exact match. Every voterName written with her real ID:
fuentes_id = PLACEHOLDER_PERSON_ID_BY_NAME["Fuentes, Jessica"]
display(real_id_rows[real_id_rows.person_id == fuentes_id].voter_name.value_counts().rename("rows").to_frame())
print("person-list names starting 'Fuentes':", elms_people[elms_people.displayName.str.startswith("Fuentes")].displayName.tolist())

In [ ]:
# Duplicate check: on the placeholder dates, does any mapped real ID ALSO appear as recorded?
# If 0, the replacement can't produce two rows for one person on one item or meeting.
mapped_ids = set(PLACEHOLDER_PERSON_ID_BY_NAME.values())
votes_as_recorded = elms_member_votes[(elms_member_votes.person_id_source == "as recorded")
                                      & elms_member_votes.person_id.isin(mapped_ids)]
votes_overlap = votes_as_recorded.event_id.isin(set(placeholder_votes.event_id)).sum()
attendance_as_recorded = elms_attendance[(elms_attendance.person_id_source == "as recorded")
                                         & elms_attendance.person_id.isin(mapped_ids)]
attendance_overlap = attendance_as_recorded.meeting_id.isin(set(placeholder_attendance.meeting_id)).sum()
print(f"as-recorded votes by mapped IDs on placeholder events: {votes_overlap}")
print(f"as-recorded attendance by mapped IDs at placeholder meetings: {attendance_overlap}")
# Any (event, person) with more than one row after replacement (unmatched placeholders excluded).
# A duplicate that is entirely 'as recorded' is already in the raw eLMS file, not caused by the table.
duplicate_votes = (elms_member_votes[elms_member_votes.person_id != PLACEHOLDER_PERSON_ID]
                   .loc[lambda frame: frame.duplicated(["event_id", "person_id"], keep=False)])
print(f"(event, person) rows appearing more than once: {len(duplicate_votes)}")
display(duplicate_votes.merge(elms_events[["event_id", "record_number", "action_date", "action_name"]], on="event_id"))

## 1.8 Undated actions (1900-01-01): every one, with what the matter file says around it

All actions in the item files (with or without votes) whose `actionDate` is the placeholder. For each:
the linked council meeting date (if any), and, for information only, the nearest dated actions before
and after it in eLMS `sort` order (a range, not a date; not used anywhere).

In [ ]:
meeting_date_by_id = elms_meetings.set_index("meeting_id").meeting_date
undated_rows = []
for matter_file in sorted((ELMS_DIR / "matters").glob("*.json")):
    matter = json.loads(matter_file.read_text())
    actions = sorted(matter.get("actions") or [], key=lambda action: action["sort"])
    for position, action in enumerate(actions):
        if action["actionDate"] not in ELMS_PLACEHOLDER_TIMESTAMPS:
            continue
        dated_before = [other["actionDate"][:10] for other in actions[:position]
                        if other["actionDate"] not in ELMS_PLACEHOLDER_TIMESTAMPS]
        dated_after = [other["actionDate"][:10] for other in actions[position + 1:]
                       if other["actionDate"] not in ELMS_PLACEHOLDER_TIMESTAMPS]
        undated_rows.append({
            "record_number": matter["recordNumber"], "action_name": action["actionName"],
            "action_by": action["actionByName"], "voter_count": len(action.get("votes") or []),
            "meeting_id": action["meetingId"], "linked_meeting_date": meeting_date_by_id.get(action["meetingId"]),
            "info_previous_dated_action_utc": dated_before[-1] if dated_before else None,
            "info_next_dated_action_utc": dated_after[0] if dated_after else None,
        })
undated_actions = pd.DataFrame(undated_rows)
undated_actions["has_votes"] = undated_actions.voter_count > 0
print(f"undated actions: {len(undated_actions)}")
display(undated_actions.groupby(["has_votes", undated_actions.linked_meeting_date.notna().rename("has_linked_meeting")])
        .size().rename("actions").to_frame())
display(undated_actions.sort_values(["has_votes", "record_number"], ascending=[False, True]).reset_index(drop=True))

In [ ]:
# The vote actions after the rule: corrected date, source flag, and the raw value kept.
display(elms_events[elms_events.action_date_source != "eLMS"]
        [["record_number", "action_name", "action_date_raw", "action_date", "action_date_source", "meeting_id"]]
        .sort_values("action_date").reset_index(drop=True))

# 2. Built tables

Each table is written by `scripts/build_tables.py` (run `python scripts/build_tables.py <table>` from the
repo root first) to `data/tables/<table>.csv`. This section reads the CSV back and shows it, so what's
checked is the file itself. Column definitions are in `PEOPLE_COLUMNS` etc. in the script.

## 2.1 `people` — one row per person in the eLMS person list

Decided 2026-10-04: everyone in the eLMS person list (no filtering); display names whitespace-stripped
only; tenure (council_start_date, years_on_council) for every person with an exact-name DataMade match,
counted to the earlier of their last DataMade end date and `AS_OF_DATE`. `ward_seats` is on hold
(DataMade `post_label` isn't reliable ward history; see the plan's future to-do).

In [ ]:
TABLES_DIR = Path("../data/tables")
# Read everything as text except the columns that are numbers/flags, so values like ward "06" stay as written.
people_table = pd.read_csv(TABLES_DIR / "people.csv", dtype=str, keep_default_na=False, na_values=[""])
for numeric_column in ["datamade_membership_count", "years_on_council", "member_vote_rows", "attendance_rows"]:
    people_table[numeric_column] = pd.to_numeric(people_table[numeric_column])
people_table["is_active"] = people_table.is_active.map({"True": True, "False": False})
print(f"{len(people_table)} rows; person_id unique: {people_table.person_id.is_unique}")
display(pd.DataFrame({"dtype": people_table.dtypes.astype(str),
                      "non_blank": people_table.notna().sum(),
                      "distinct": people_table.nunique()}))

In [ ]:
# Categorical columns: every value and its count.
for column in ["is_active", "start_source", "datamade_membership_count"]:
    display(people_table[column].value_counts(dropna=False).rename("people").to_frame())
display(pd.crosstab(people_table.elms_ward.where(people_table.elms_ward.str.isdigit(), people_table.elms_ward)
                    .map(lambda ward: "a ward number" if ward.isdigit() else ward), people_table.is_active))

In [ ]:
# Every row (123). years_on_council rounded to 1 decimal for display only (the CSV is unrounded).
pd.set_option("display.max_rows", 300)
display(people_table.drop(columns=["photo_url", "site", "display_name_raw"])
        .assign(years_on_council=people_table.years_on_council.round(1)).reset_index(drop=True))

**Cross-checks**

In [ ]:
# 1. Active people vs §1.1 (50 wards, one active alder each).
active_wards = people_table[people_table.is_active & people_table.elms_ward.str.isdigit()].elms_ward
print(f"active people with a ward number: {len(active_wards)}; distinct wards: {active_wards.nunique()}")

# 2. Names that differ from the raw displayName only by surrounding whitespace.
print(f"display names changed by stripping: {(people_table.display_name != people_table.display_name_raw).sum()}")

# 3. People without a DataMade match (no tenure).
display(people_table[people_table.start_source == "no DataMade match"]
        [["display_name", "elms_ward", "is_active", "member_vote_rows", "attendance_rows"]])

# 4. People with more than one DataMade record: tenure ignores the gaps (decision 1.2).
display(people_table[people_table.datamade_membership_count > 1]
        [["display_name", "elms_ward", "is_active", "council_start_date", "datamade_last_end_date",
          "years_on_council"]])

In [ ]:
# 5. Row counts vs the rows loaded at the top of this notebook (after the placeholder replacement).
#    Every person_id in the vote/attendance rows that is NOT in `people`:
for name, frame in [("member votes", elms_member_votes), ("attendance", elms_attendance)]:
    not_in_people = frame[~frame.person_id.isin(people_table.person_id)]
    print(f"{name}: {len(frame):,} rows; in people: {people_table[{'member votes': 'member_vote_rows', 'attendance': 'attendance_rows'}[name]].sum():,}; "
          f"person_id not in people: {len(not_in_people)}")
    display(not_in_people.groupby(["person_id", "voter_name"]).size().rename("rows").to_frame())

# 6. People in the list with no vote rows at all.
display(people_table[people_table.member_vote_rows == 0][["display_name", "elms_ward", "is_active", "attendance_rows"]])

In [ ]:
# Shared reader for the rest of section 2: everything as text (so "06" stays "06", blank stays blank),
# then the listed columns converted to numbers / True-False.
def read_table(name, numeric_columns=(), boolean_columns=()):
    table = pd.read_csv(TABLES_DIR / f"{name}.csv", dtype=str, keep_default_na=False, na_values=[""])
    for column in numeric_columns:
        table[column] = pd.to_numeric(table[column])
    for column in boolean_columns:
        table[column] = table[column].map({"True": True, "False": False})
    return table

def describe_table(table):
    """dtype, non-blank count and distinct count per column."""
    display(pd.DataFrame({"dtype": table.dtypes.astype(str), "non_blank": table.notna().sum(),
                          "distinct": table.nunique()}))

def show_values(table, columns):
    """Every value and its count, for each listed column (blank shown as NaN)."""
    for column in columns:
        display(table[column].value_counts(dropna=False).rename("rows").to_frame())

## 2.2 `meetings` — one row per eLMS meeting file

All 88 files (all City Council, including cancelled ones; nothing filtered). `vote_event_count` and
`roll_call_event_count` count the `vote_events` rows linked to each meeting (after the date match).

In [ ]:
meetings_table = read_table("meetings",
    numeric_columns=["attendance_roll_call_count", "attendance_member_count", "agenda_item_count",
                     "vote_event_count", "roll_call_event_count"],
    boolean_columns=["has_attendance"])
print(f"{len(meetings_table)} rows; meeting_id unique: {meetings_table.meeting_id.is_unique}")
describe_table(meetings_table)
show_values(meetings_table, ["body", "status", "attendance_roll_call_names", "attendance_member_count"])
display(pd.crosstab(meetings_table.status, [meetings_table.has_attendance, meetings_table.roll_call_event_count > 0],
                    margins=True))

In [ ]:
# Every row (88).
display(meetings_table.drop(columns=["meeting_datetime_utc"]))

## 2.3 `attendance` — one row per meeting × attendance roll call × member

Raw roster rows, with the placeholder IDs replaced (decision 1.7; raw ID in `person_id_raw`). One raw
duplicate is kept (`KNOWN_DUPLICATE_ATTENDANCE` in the script).

In [ ]:
attendance_table = read_table("attendance")
print(f"{len(attendance_table):,} rows")
describe_table(attendance_table)
show_values(attendance_table, ["roll_call_name", "status", "person_id_source"])
print("rows per meeting:")
display(attendance_table.groupby("meeting_date").size().value_counts().rename("meetings").to_frame())
display(attendance_table[attendance_table.duplicated(["meeting_id", "roll_call_name", "person_id"], keep=False)])
display(attendance_table.head(5))

## 2.4 `vote_events` — one row per action (any body) that has a per-member roster

Every such action in the item files (17,153), nothing filtered. Corrected columns sit beside the raw
ones, each with a `*_source` column:

| Column | Raw column | Rule |
|---|---|---|
| `action_date` | `action_date_raw` | Chicago date; 1900-01-01 → linked meeting's date (1.8) |
| `meeting_id` | `meeting_id_raw` | blank → the one council meeting that date (`MATCH_MEETING_BY_DATE`) |
| `action_by` | `action_by_raw` | blank + 50 voters + council meeting → "City Council" (1.6) |
| `roster_kind` | — | first-match rules (§1.4d) |

In [ ]:
count_columns = ["count_yea", "count_nay", "count_absent", "count_not_voting", "count_present",
                 "count_recused", "count_rising_vote", "count_vacant"]
vote_events_table = read_table("vote_events", numeric_columns=["voter_count", *count_columns],
                               boolean_columns=["meeting_has_file"])
print(f"{len(vote_events_table):,} rows; event_id unique: {vote_events_table.event_id.is_unique}")
describe_table(vote_events_table)
show_values(vote_events_table, ["roster_kind", "action_date_source", "meeting_id_source", "meeting_has_file",
                                "action_by_source", "matter_type", "agreed_calendar", "routine", "voter_count"])

In [ ]:
# action_by: raw vs corrected.
display(pd.crosstab(vote_events_table.action_by_raw.fillna("(blank)"), vote_events_table.action_by_source))
# OPEN QUESTION check (see RELABEL rule in the script): blank actionByName, 50 voters, meeting only from the
# date match — not relabeled under the decision as written.
not_relabeled = vote_events_table[vote_events_table.action_by_raw.isna() & (vote_events_table.voter_count == 50)
                                  & (vote_events_table.action_by_source == "eLMS")]
print(f"blank actionByName, 50 voters, NOT relabeled: {len(not_relabeled)}")
display(not_relabeled[["action_date", "record_number", "action_name", "meeting_id_source", "meeting_has_file"]])

In [ ]:
# Same results as section 1? roster_kind counts (§1.4d) and the date fixes (§1.8).
print("roster_kind:", vote_events_table.roster_kind.value_counts().to_dict())
print("meeting_id_source:", vote_events_table.meeting_id_source.value_counts().to_dict())
print("action_date_source:", vote_events_table.action_date_source.value_counts().to_dict())
# Rosters shared by more than one event (identical who-and-how): how many events per signature.
signature_sizes = vote_events_table.roster_signature.value_counts()
print("events sharing their exact roster with at least one other event:",
      int(signature_sizes[signature_sizes > 1].sum()), "in", int((signature_sizes > 1).sum()), "rosters")
display(vote_events_table.head(3).T)

## 2.5 `member_votes` — one row per vote event × roster row

Raw values (decision 1.3), placeholder IDs replaced (1.7). One raw duplicate kept
(`KNOWN_DUPLICATE_MEMBER_VOTES` in the script).

In [ ]:
member_votes_table = read_table("member_votes")
print(f"{len(member_votes_table):,} rows")
describe_table(member_votes_table)
show_values(member_votes_table, ["vote", "person_id_source"])
display(member_votes_table[member_votes_table.duplicated(["event_id", "person_id"], keep=False)
                           & (member_votes_table.person_id_source != "placeholder, unmatched")])

## 2.6 How the tables link

```
people.person_id ──< member_votes.person_id      member_votes.event_id >── vote_events.event_id
people.person_id ──< attendance.person_id        vote_events.meeting_id >── meetings.meeting_id
                                                  attendance.meeting_id  >── meetings.meeting_id
```

For each link: rows whose key is missing on the other side (both directions). Then two results from
section 1 / the plan, rebuilt from the tables alone.

In [ ]:
def link_check(left, left_key, right, right_key, label):
    left_keys, right_keys = set(left[left_key].dropna()), set(right[right_key].dropna())
    return {"link": label,
            "left rows with blank key": int(left[left_key].isna().sum()),
            "left rows with no match on right": int((~left[left_key].isin(right_keys) & left[left_key].notna()).sum()),
            "right keys never used on left": len(right_keys - left_keys)}

display(pd.DataFrame([
    link_check(member_votes_table, "event_id", vote_events_table, "event_id", "member_votes → vote_events"),
    link_check(member_votes_table, "person_id", people_table, "person_id", "member_votes → people"),
    link_check(attendance_table, "person_id", people_table, "person_id", "attendance → people"),
    link_check(attendance_table, "meeting_id", meetings_table, "meeting_id", "attendance → meetings"),
    link_check(vote_events_table, "meeting_id", meetings_table, "meeting_id", "vote_events → meetings"),
]))

In [ ]:
# vote_events whose meeting_id has no meeting file / is blank, by source (expected: pre-term + the 2025-10-16 id,
# and the 2 blank 2023-04-19 events — section 1 handoff).
unlinked = vote_events_table[~vote_events_table.meeting_has_file]
display(unlinked.groupby([unlinked.action_date.str[:7], "meeting_id_source"]).size().rename("events").to_frame())
# member_votes person_ids not in people (expected: the Vacant IDs).
display(member_votes_table[~member_votes_table.person_id.isin(people_table.person_id)]
        .groupby(["person_id", "voter_name"]).size().rename("rows").to_frame())

In [ ]:
# Rebuilt from the tables: 2025 budget SO2024-0013682 (plan: 27 yes / 23 no).
budget = vote_events_table[vote_events_table.record_number == "SO2024-0013682"]
display(budget[["action_date", "action_by", "action_name", "roster_kind", "count_yea", "count_nay", "count_absent"]])

# Rebuilt from the tables: §1.4d absence — share of each council meeting's roll-call items an alder was Absent on,
# absent from the meeting if share > ABSENT_MEETING_SHARE_THRESHOLD (§1.4d: 52 absent / 3,151 present).
council_meeting_ids_table = set(meetings_table.loc[meetings_table.body == "City Council", "meeting_id"])
roll_call_events_table = vote_events_table[(vote_events_table.roster_kind == "roll call")
                                           & vote_events_table.meeting_id.isin(council_meeting_ids_table)]
per_alder_meeting = (member_votes_table.merge(roll_call_events_table[["event_id", "meeting_id"]], on="event_id")
                     .assign(is_absent=lambda frame: frame.vote == "Absent")
                     .groupby(["meeting_id", "person_id"]).is_absent.mean())
print(f"alder-meetings: {len(per_alder_meeting)}; absent from meeting: "
      f"{(per_alder_meeting > ABSENT_MEETING_SHARE_THRESHOLD).value_counts().to_dict()}")